# CHOMP: Covariant Hamiltonian Optimization for Motion Planning
## A Hands-On Tutorial with a 3R Planar Robotic Arm

This notebook provides a complete, from-scratch implementation of the **CHOMP** algorithm — a powerful gradient-based trajectory optimizer for robotic motion planning.

**What you'll learn:**
1. How CHOMP formulates motion planning as continuous optimization
2. The mathematics: objective functionals, functional gradients, covariant updates
3. A working implementation for a 3-link planar arm — first without obstacles, then with obstacles

**Prerequisites:** Linear algebra, basic calculus, familiarity with forward kinematics.

**Reference:** Zucker et al., *CHOMP: Covariant Hamiltonian Optimization for Motion Planning*, IJRR 2013.

---
## 1. The Core Idea

Most classical planners (RRT, PRM) treat motion planning as a **graph search** problem. CHOMP takes a fundamentally different approach: it treats it as a **continuous optimization** problem.

A trajectory $\xi$ is a smooth function mapping time to the robot's configuration (joint angles):

$$\xi : [0, 1] \to \mathcal{Q} \subset \mathbb{R}^d$$

where $d$ is the number of degrees of freedom. Since $\xi$ is a function, our cost must be an **objective functional** — a function that takes another function as input and returns a scalar:

$$U[\xi] = F_{\text{obs}}[\xi] + \lambda \, F_{\text{smooth}}[\xi]$$

| Term | Role | Intuition |
|------|------|-----------|
| $F_{\text{obs}}$ | Obstacle cost | Penalizes proximity to obstacles |
| $F_{\text{smooth}}$ | Smoothness cost | Penalizes jerky, high-velocity motion |
| $\lambda$ | Trade-off weight | Balances safety vs. smoothness |

## 2. Smoothness Functional

The smoothness cost measures the total dynamical "effort" of the trajectory:

$$F_{\text{smooth}}[\xi] = \frac{1}{2} \int_0^1 \left\| \frac{d\xi}{dt} \right\|^2 dt$$

This is the integral of squared velocity over time. Minimizing it penalizes jerky movements and naturally prefers **constant-velocity, straight-line** paths in joint space (the minimum-energy trajectory between two configurations).

## 3. Obstacle Functional

The obstacle cost is computed as a **workspace integral** over the robot's body:

$$F_{\text{obs}}[\xi] = \int_0^1 \int_B c\big(x(\xi(t), u)\big) \left\| \frac{d}{dt} x(\xi(t), u) \right\| \, du \, dt$$

where:
- $B$ is the robot's body, parameterized by $u$
- $x(\xi(t), u)$ is the workspace position of body point $u$ at configuration $\xi(t)$ (via forward kinematics)
- $c(\cdot)$ is a cost field penalizing proximity to obstacles
- $\left\| \frac{d}{dt} x \right\|$ is the **arc-length weighting** — it makes the cost invariant to re-timing

The cost function $c$ is derived from the **signed distance field** $d(x)$ (negative inside obstacles, positive outside):

$$c(d) = \begin{cases} -d + \frac{\varepsilon}{2} & \text{if } d < 0 \\ \frac{1}{2\varepsilon}(d - \varepsilon)^2 & \text{if } 0 \le d \le \varepsilon \\ 0 & \text{if } d > \varepsilon \end{cases}$$

where $\varepsilon$ is the safety margin. The cost is:
- **Large** inside obstacles ($d < 0$)
- **Smoothly decaying** in the safety margin ($0 \le d \le \varepsilon$)
- **Zero** far from obstacles ($d > \varepsilon$)

## 4. Functional Gradients

To minimize $U[\xi]$ via gradient descent, we need the **functional gradient** $\bar{\nabla}U$. Since $\xi$ is a function (not a vector), we use the **Euler-Lagrange equation** from the calculus of variations.

For a functional of the form $\int v(\xi, \xi') \, dt$, the direction of steepest descent is:

$$\bar{\nabla}U[\xi] = \frac{\partial v}{\partial \xi} - \frac{d}{dt}\frac{\partial v}{\partial \xi'}$$

This tells us how to perturb the **entire continuous trajectory** to maximally decrease the cost.

## 5. Covariant Gradients and the Metric $A$

**The problem with naive gradient descent:** If the optimizer detects a collision at one waypoint, a standard (Euclidean) gradient update would aggressively yank *just that waypoint* away from the obstacle — creating a sharp kink in the trajectory.

**CHOMP's solution:** Define a custom **Riemannian metric** $A$ that measures perturbation size in terms of *physical dynamics* (acceleration/velocity) rather than Euclidean distance. The **covariant gradient** is:

$$\tilde{\nabla}U = A^{-1} \bar{\nabla}U$$

$A^{-1}$ acts as a **smoothing operator**: it distributes a localized obstacle-avoidance push smoothly across neighboring waypoints, preventing kinks.

## 6. Discretization: The $K$ Matrix

To make this computable, we discretize $\xi$ into $n$ interior waypoints with fixed endpoints:

$$\xi = [q_1, q_2, \ldots, q_n] \quad \text{with fixed } q_0 \text{ (start) and } q_{n+1} \text{ (goal)}$$

Velocities are approximated by **finite differences**. The matrix $K \in \mathbb{R}^{(n+1) \times n}$ computes these differences:

$$K = \begin{bmatrix} 1 & 0 & 0 & \cdots & 0 \\ -1 & 1 & 0 & \cdots & 0 \\ 0 & -1 & 1 & \cdots & 0 \\ \vdots & & \ddots & \ddots & \vdots \\ 0 & 0 & \cdots & -1 & 1 \\ 0 & 0 & \cdots & 0 & -1 \end{bmatrix}$$

The full velocity vector is $K\xi + e$, where $e$ captures boundary contributions ($e_0 = -q_0$, $e_n = q_{n+1}$).

The **metric matrix** and **smoothness cost** become:

$$A = K^T K \quad \text{(tridiagonal: 2 on diagonal, -1 on off-diagonals)}$$

$$F_{\text{smooth}} = \frac{1}{2} \|K\xi + e\|^2 = \frac{1}{2}\xi^T A \xi + \xi^T b + \text{const}$$

where $b = K^T e$.

## 7. The CHOMP Update Rule

Putting it all together, the iterative update is:

$$\boxed{\xi_{i+1} = \xi_i - \frac{1}{\eta} A^{-1} \bar{\nabla}U[\xi_i]}$$

where:
- $\eta$ is the step size (learning rate)
- $A^{-1}$ is the smoothing matrix (inverse of the metric)
- $\bar{\nabla}U = \nabla F_{\text{obs}} + \lambda (A\xi + b)$ is the total functional gradient

**Note on Hamiltonian Monte Carlo:** CHOMP can optionally incorporate HMC to escape local minima by adding random momentum to the trajectory. We focus on the gradient descent version in this tutorial.

---
# Implementation

Now let's implement everything step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import animation
from matplotlib.colors import Normalize
from scipy.interpolate import RegularGridInterpolator
from IPython.display import HTML

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12
np.random.seed(42)

In [ ]:
# === 3R Planar Arm Parameters ===
L1, L2, L3 = 1.0, 0.8, 0.6       # Link lengths
LINK_LENGTHS = [L1, L2, L3]
DOF = 3                             # Degrees of freedom

# === CHOMP Parameters ===
N_WAYPOINTS = 50                    # Number of interior waypoints
LAMBDA = 10.0                       # Smoothness weight
ETA = 100.0                         # Step size (learning rate)
MAX_ITER = 200                      # Maximum iterations
EPSILON = 0.3                       # Obstacle safety margin

# === Workspace Parameters ===
WS_XLIM = (-3.0, 3.0)
WS_YLIM = (-3.0, 3.0)
GRID_RESOLUTION = 0.02             # SDF grid cell size
N_BODY_POINTS = 10                 # Body points sampled per link

---
## 8. The 3R Planar Arm

Our robot is a **3-revolute (3R) planar arm** — three rigid links connected by revolute joints, all moving in the 2D plane.

**Configuration:** $q = (\theta_1, \theta_2, \theta_3)$

**Forward kinematics** — joint positions:

| Point | Position |
|-------|----------|
| Base | $(0, 0)$ |
| Joint 1 | $(L_1 \cos\theta_1,\; L_1 \sin\theta_1)$ |
| Joint 2 | $p_1 + (L_2 \cos(\theta_1+\theta_2),\; L_2 \sin(\theta_1+\theta_2))$ |
| End-effector | $p_2 + (L_3 \cos(\theta_1+\theta_2+\theta_3),\; L_3 \sin(\theta_1+\theta_2+\theta_3))$ |

For CHOMP, we also need body points **along each link** (not just at joints) and the **Jacobian** at each body point — relating joint velocity $\dot{q}$ to workspace velocity of that point.

In [ ]:
def forward_kinematics(q):
    """
    Compute joint and end-effector positions for the 3R planar arm.
    
    Args:
        q: (3,) array of joint angles [theta1, theta2, theta3]
    Returns:
        positions: (4, 2) array — [base, joint1, joint2, end_effector]
    """
    positions = np.zeros((4, 2))
    cum_angle = 0.0
    for i in range(3):
        cum_angle += q[i]
        positions[i + 1] = positions[i] + LINK_LENGTHS[i] * np.array([np.cos(cum_angle), np.sin(cum_angle)])
    return positions

In [ ]:
def body_point_position(q, link_idx, u):
    """
    Position of a point at parameter u in [0, 1] along link `link_idx`.
    u=0 is the start of the link, u=1 is the end.
    """
    positions = forward_kinematics(q)
    p_start = positions[link_idx]      # start of link
    p_end = positions[link_idx + 1]    # end of link
    return p_start + u * (p_end - p_start)


def compute_jacobian(q, link_idx, u):
    """
    Compute the 2x3 Jacobian for a body point at parameter u on link `link_idx`.
    
    J maps joint velocities dq/dt to workspace velocity dx/dt of this body point.
    """
    J = np.zeros((2, DOF))
    
    # Cumulative angles for each link
    cum_angles = np.cumsum(q)
    
    # Joint j affects this body point only if j <= link_idx
    for j in range(link_idx + 1):
        # Full links from j to link_idx - 1
        for m in range(j, link_idx):
            J[0, j] += -LINK_LENGTHS[m] * np.sin(cum_angles[m])
            J[1, j] +=  LINK_LENGTHS[m] * np.cos(cum_angles[m])
        # Partial link (link_idx) with parameter u
        J[0, j] += -u * LINK_LENGTHS[link_idx] * np.sin(cum_angles[link_idx])
        J[1, j] +=  u * LINK_LENGTHS[link_idx] * np.cos(cum_angles[link_idx])
    
    return J


def compute_all_body_data(q, n_body_points=N_BODY_POINTS):
    """
    Compute all body point positions and Jacobians for a single configuration.
    FK and cumulative angles are computed once, then reused for all body points.
    
    Returns:
        positions: (3 * n_body_points, 2) array of body point positions
        jacobians: (3 * n_body_points, 2, DOF) array of Jacobians
    """
    # FK once
    joint_positions = forward_kinematics(q)
    cum_angles = np.cumsum(q)
    
    n_total = 3 * n_body_points
    positions = np.zeros((n_total, 2))
    jacobians = np.zeros((n_total, 2, DOF))
    
    # Precompute sin/cos of cumulative angles (used by all body points)
    sin_ca = np.sin(cum_angles)
    cos_ca = np.cos(cum_angles)
    
    # Precompute per-link direction vectors (link_start -> link_end)
    link_dirs = np.zeros((3, 2))
    for k in range(3):
        link_dirs[k] = joint_positions[k + 1] - joint_positions[k]
    
    # Precompute cumulative Jacobian contributions for full links
    # full_J_contrib[link_idx, j] = sum over m in [j, link_idx) of L_m * [-sin, cos]
    # This avoids recomputing the inner loop for each body point
    full_J_contrib = np.zeros((3, DOF, 2))  # [link_idx, joint_j, (x,y)]
    for link_idx in range(3):
        for j in range(link_idx + 1):
            for m in range(j, link_idx):
                full_J_contrib[link_idx, j, 0] += -LINK_LENGTHS[m] * sin_ca[m]
                full_J_contrib[link_idx, j, 1] +=  LINK_LENGTHS[m] * cos_ca[m]
    
    idx = 0
    for link_idx in range(3):
        # Partial link Jacobian contribution (same for all u, just scaled)
        partial_x = -LINK_LENGTHS[link_idx] * sin_ca[link_idx]
        partial_y =  LINK_LENGTHS[link_idx] * cos_ca[link_idx]
        
        for bp in range(n_body_points):
            u = (bp + 0.5) / n_body_points
            
            # Position: interpolate along link
            positions[idx] = joint_positions[link_idx] + u * link_dirs[link_idx]
            
            # Jacobian: full-link contributions + u-scaled partial link
            for j in range(link_idx + 1):
                jacobians[idx, 0, j] = full_J_contrib[link_idx, j, 0] + u * partial_x
                jacobians[idx, 1, j] = full_J_contrib[link_idx, j, 1] + u * partial_y
            
            idx += 1
    
    return positions, jacobians

In [ ]:
def plot_arm(ax, q, color='steelblue', alpha=1.0, linewidth=3, markersize=8, label=None):
    """Draw the 3R arm on the given axes."""
    positions = forward_kinematics(q)
    ax.plot(positions[:, 0], positions[:, 1], 'o-',
            color=color, linewidth=linewidth, markersize=markersize,
            alpha=alpha, solid_capstyle='round', label=label)
    # Mark base with a square
    ax.plot(0, 0, 's', color='black', markersize=10, zorder=5)

In [ ]:
# Visualize the arm in several configurations
test_configs = [
    np.array([0.5, 0.3, 0.2]),
    np.array([1.2, -0.8, 0.5]),
    np.array([np.pi/4, np.pi/3, -np.pi/6]),
    np.array([2.0, -1.0, 0.8]),
]
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

fig, ax = plt.subplots(figsize=(8, 8))
for q_test, c in zip(test_configs, colors):
    plot_arm(ax, q_test, color=c, alpha=0.8,
             label=f'q = ({q_test[0]:.1f}, {q_test[1]:.1f}, {q_test[2]:.1f})')

ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-2.8, 2.8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
ax.set_title('3R Planar Arm — Test Configurations')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.tight_layout()
plt.show()

---
## 9. Trajectory Representation

We discretize the trajectory into $n$ interior waypoints with fixed start and goal:

$$\xi = [q_1, q_2, \ldots, q_n] \in \mathbb{R}^{n \times d}$$

The **initial trajectory** is a straight-line interpolation in joint space — the simplest guess.

In [ ]:
def init_trajectory(q_start, q_goal, n_waypoints):
    """Create a straight-line trajectory in joint space (excludes endpoints)."""
    xi = np.zeros((n_waypoints, DOF))
    for d in range(DOF):
        xi[:, d] = np.linspace(q_start[d], q_goal[d], n_waypoints + 2)[1:-1]
    return xi


def full_trajectory(xi, q_start, q_goal):
    """Prepend start and append goal to the interior waypoints."""
    return np.vstack([q_start, xi, q_goal])

In [ ]:
def plot_trajectory_workspace(xi, q_start, q_goal, ax=None, obstacles=None,
                              n_arms=8, title='Trajectory in Workspace'):
    """
    Visualize trajectory: draw the arm at several waypoints and trace the end-effector path.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    
    traj = full_trajectory(xi, q_start, q_goal)
    n_total = len(traj)
    
    # Draw obstacles if any
    if obstacles is not None:
        for obs in obstacles:
            circle = plt.Circle(obs['center'], obs['radius'],
                              color='red', alpha=0.3, zorder=2)
            ax.add_patch(circle)
            circle_edge = plt.Circle(obs['center'], obs['radius'],
                                    fill=False, edgecolor='red', linewidth=2, zorder=2)
            ax.add_patch(circle_edge)
    
    # Draw arm at evenly spaced waypoints
    indices = np.linspace(0, n_total - 1, n_arms, dtype=int)
    for idx in indices:
        alpha = 0.15 + 0.75 * (idx / (n_total - 1))
        plot_arm(ax, traj[idx], color='steelblue', alpha=alpha, linewidth=2, markersize=5)
    
    # Highlight start and goal
    plot_arm(ax, q_start, color='green', linewidth=3, markersize=8, label='Start')
    plot_arm(ax, q_goal, color='red', linewidth=3, markersize=8, label='Goal')
    
    # End-effector path
    ee_path = np.array([forward_kinematics(q)[3] for q in traj])
    ax.plot(ee_path[:, 0], ee_path[:, 1], '-', color='navy', linewidth=2, alpha=0.8, label='EE path')
    
    ax.set_xlim(-2.8, 2.8)
    ax.set_ylim(-2.8, 2.8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=10)
    ax.set_title(title)
    return ax


def plot_trajectory_joint_space(xi, q_start, q_goal, title_prefix=''):
    """Plot each joint angle vs. waypoint index."""
    traj = full_trajectory(xi, q_start, q_goal)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for d in range(DOF):
        axes[d].plot(traj[:, d], 'o-', markersize=2, color='steelblue')
        axes[d].set_title(f'{title_prefix}Joint {d+1} ($\\theta_{d+1}$)')
        axes[d].set_xlabel('Waypoint index')
        axes[d].set_ylabel('Angle (rad)')
        axes[d].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Define start and goal for our planning problems
q_start = np.array([0.5, 0.3, 0.2])
q_goal = np.array([2.2, -0.8, 1.0])

xi_straight = init_trajectory(q_start, q_goal, N_WAYPOINTS)

fig, ax = plt.subplots(figsize=(8, 8))
plot_trajectory_workspace(xi_straight, q_start, q_goal, ax=ax,
                          title='Initial Straight-Line Trajectory')
plt.tight_layout()
plt.show()

plot_trajectory_joint_space(xi_straight, q_start, q_goal, title_prefix='Initial: ')

---
## 10. Smoothness Cost & the $A$ Matrix

### Building $K$ and $A$

The finite-difference matrix $K \in \mathbb{R}^{(n+1) \times n}$ computes velocities between consecutive waypoints. For $n$ interior waypoints and $n+1$ intervals:

$$v_i = q_{i+1} - q_i \quad \text{for } i = 0, 1, \ldots, n$$

In matrix form: $v = K\xi + e$

The smoothness cost becomes:

$$F_{\text{smooth}} = \frac{1}{2}\|K\xi + e\|^2$$

And its gradient is:

$$\nabla_{\xi} F_{\text{smooth}} = A\xi + b, \quad \text{where } A = K^TK, \; b = K^Te$$

In [ ]:
def build_smoothness_matrices(n_waypoints, q_start, q_goal):
    """
    Build the finite difference matrix K, metric A = K^T K,
    its inverse A_inv, and boundary vectors e and b = K^T e.
    
    K is (n+1) x n for first-order (velocity) differences.
    """
    n = n_waypoints
    
    # K matrix: (n+1) x n
    # Row 0:   v_0 = q_1 - q_0       -> K[0,0] = 1, boundary e[0] = -q_0
    # Row i:   v_i = q_{i+1} - q_i   -> K[i,i] = 1, K[i,i-1] = -1
    # Row n:   v_n = q_{n+1} - q_n   -> K[n,n-1] = -1, boundary e[n] = q_{n+1}
    K = np.zeros((n + 1, n))
    K[0, 0] = 1.0
    for i in range(1, n):
        K[i, i] = 1.0
        K[i, i - 1] = -1.0
    K[n, n - 1] = -1.0
    
    # Boundary vector e: (n+1) x DOF
    e = np.zeros((n + 1, DOF))
    e[0, :] = -q_start
    e[n, :] = q_goal
    
    # Metric matrix A = K^T K  (n x n, symmetric positive definite)
    A = K.T @ K
    
    # Precompute inverse (n is small, direct inversion is fine)
    A_inv = np.linalg.inv(A)
    
    # Boundary contribution to gradient: b = K^T e  (n x DOF)
    b = K.T @ e
    
    return K, A, A_inv, e, b

In [ ]:
# Build matrices and visualize
K, A, A_inv, e, b = build_smoothness_matrices(N_WAYPOINTS, q_start, q_goal)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im0 = axes[0].imshow(A, cmap='RdBu_r', aspect='equal')
axes[0].set_title('Metric Matrix $A = K^T K$ (tridiagonal)', fontsize=13)
axes[0].set_xlabel('Waypoint j')
axes[0].set_ylabel('Waypoint i')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(A_inv, cmap='RdBu_r', aspect='equal')
axes[1].set_title('Smoothing Matrix $A^{-1}$ (dense!)', fontsize=13)
axes[1].set_xlabel('Waypoint j')
axes[1].set_ylabel('Waypoint i')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

print(f'A is tridiagonal with diagonal = {A[0,0]:.0f}, off-diagonal = {A[0,1]:.0f}')
print(f'A_inv is dense — a push at one waypoint spreads across all others')
print(f'A shape: {A.shape}, A_inv shape: {A_inv.shape}')

### Understanding $A^{-1}$ as a Smoothing Operator

Notice that $A$ is **sparse** (tridiagonal), but $A^{-1}$ is **dense**. This is the key insight:

- A raw gradient push at waypoint $i$ would only affect waypoint $i$ (Euclidean gradient)
- After multiplying by $A^{-1}$ (covariant gradient), the push is **distributed smoothly** across all waypoints
- The influence decays linearly away from waypoint $i$ — like pulling a rope at one point

In [ ]:
def compute_smoothness_cost(xi, K, e):
    """F_smooth = (1/2) sum_d ||K @ xi_d + e_d||^2"""
    cost = 0.0
    for d in range(DOF):
        vel = K @ xi[:, d] + e[:, d]
        cost += 0.5 * np.dot(vel, vel)
    return cost


def compute_smoothness_gradient(xi, A, b):
    """Gradient of F_smooth w.r.t. xi: A @ xi + b. Shape: (n, DOF)."""
    return A @ xi + b

In [ ]:
# Verify gradient with finite differences
xi_test = xi_straight + 0.1 * np.random.randn(*xi_straight.shape)
grad_analytical = compute_smoothness_gradient(xi_test, A, b)

eps_fd = 1e-5
grad_numerical = np.zeros_like(xi_test)
for i in range(xi_test.shape[0]):
    for d in range(DOF):
        xi_plus = xi_test.copy(); xi_plus[i, d] += eps_fd
        xi_minus = xi_test.copy(); xi_minus[i, d] -= eps_fd
        grad_numerical[i, d] = (compute_smoothness_cost(xi_plus, K, e) -
                                compute_smoothness_cost(xi_minus, K, e)) / (2 * eps_fd)

rel_error = np.linalg.norm(grad_analytical - grad_numerical) / (np.linalg.norm(grad_numerical) + 1e-10)
print(f'Smoothness gradient verification:')
print(f'  Relative error: {rel_error:.2e} (should be < 1e-6)')
print(f'  PASS' if rel_error < 1e-5 else f'  FAIL')

---
## 11. Case 1: CHOMP Without Obstacles

With only the smoothness cost, CHOMP should drive any trajectory toward the **minimum-velocity path** — a straight line in joint space.

To test this, we start from a **deliberately perturbed** (non-straight) trajectory and watch CHOMP smooth it out.

In [ ]:
# Create a perturbed initial trajectory with sinusoidal bumps
xi_perturbed = init_trajectory(q_start, q_goal, N_WAYPOINTS).copy()
t = np.linspace(0, 1, N_WAYPOINTS)
xi_perturbed[:, 0] += 0.5 * np.sin(2 * np.pi * t)
xi_perturbed[:, 1] -= 0.4 * np.sin(4 * np.pi * t)
xi_perturbed[:, 2] += 0.3 * np.sin(3 * np.pi * t)

In [ ]:
def chomp_no_obstacles(xi_init, q_start, q_goal, n_waypoints,
                       lam=LAMBDA, eta=ETA, max_iter=MAX_ITER):
    """
    CHOMP with only smoothness cost.
    
    Update: xi <- xi - (1/eta) * A_inv @ (lam * smooth_gradient)
    """
    K, A, A_inv, e, b_vec = build_smoothness_matrices(n_waypoints, q_start, q_goal)
    
    xi = xi_init.copy()
    cost_history = []
    trajectory_history = [xi.copy()]
    
    for iteration in range(max_iter):
        cost = compute_smoothness_cost(xi, K, e)
        cost_history.append(cost)
        
        grad_smooth = compute_smoothness_gradient(xi, A, b_vec)
        
        # Covariant update
        xi -= (lam / eta) * (A_inv @ grad_smooth)
        
        if iteration % 10 == 0:
            trajectory_history.append(xi.copy())
    
    trajectory_history.append(xi.copy())
    return xi, cost_history, trajectory_history

In [ ]:
xi_smooth, costs_smooth, traj_hist_smooth = chomp_no_obstacles(
    xi_perturbed, q_start, q_goal, N_WAYPOINTS)

# Cost convergence
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(costs_smooth, 'steelblue', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Smoothness Cost $F_{smooth}$')
ax.set_title('CHOMP Without Obstacles: Cost Convergence')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Workspace comparison: initial vs optimized vs evolution
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

plot_trajectory_workspace(xi_perturbed, q_start, q_goal, ax=axes[0],
                          title='Initial (Perturbed)')
plot_trajectory_workspace(xi_smooth, q_start, q_goal, ax=axes[1],
                          title='Optimized (Smooth)')

# Evolution overlay — end-effector paths from intermediate trajectories
ax = axes[2]
plot_arm(ax, q_start, color='green', linewidth=3, label='Start')
plot_arm(ax, q_goal, color='red', linewidth=3, label='Goal')
cmap = plt.cm.viridis
for j, xi_hist in enumerate(traj_hist_smooth):
    traj = full_trajectory(xi_hist, q_start, q_goal)
    ee = np.array([forward_kinematics(q)[3] for q in traj])
    color = cmap(j / max(len(traj_hist_smooth) - 1, 1))
    ax.plot(ee[:, 0], ee[:, 1], '-', color=color, alpha=0.6, linewidth=1.5)
ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-2.8, 2.8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
ax.set_title('Evolution (dark → light = early → late)')

plt.tight_layout()
plt.show()

In [ ]:
# Joint space: before vs after
traj_init = full_trajectory(xi_perturbed, q_start, q_goal)
traj_opt = full_trajectory(xi_smooth, q_start, q_goal)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for d in range(DOF):
    axes[d].plot(traj_init[:, d], '--', color='coral', linewidth=2, label='Initial')
    axes[d].plot(traj_opt[:, d], '-', color='steelblue', linewidth=2, label='Optimized')
    axes[d].set_title(f'Joint {d+1} ($\\theta_{d+1}$)')
    axes[d].set_xlabel('Waypoint index')
    axes[d].set_ylabel('Angle (rad)')
    axes[d].grid(True, alpha=0.3)
    axes[d].legend(fontsize=9)
plt.suptitle('Joint Space: Bumpy Initial → Smooth Straight Line', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Without obstacles, CHOMP drives the trajectory to a straight line in joint space.')
print(f'Final smoothness cost: {costs_smooth[-1]:.4f}')

---
## 12. Obstacle Cost

Now we add obstacles. We need:
1. A **signed distance field (SDF)** for the workspace
2. The **cost function** $c(d)$ and its derivative
3. The **obstacle gradient** $\nabla_q F_{\text{obs}}$ using the chain rule through forward kinematics

### Signed Distance Field

For circular obstacles, the signed distance is analytical:

$$d_i(x) = \|x - c_i\| - r_i$$

For multiple obstacles: $d(x) = \min_i d_i(x)$

In [ ]:
# Define obstacles (circular)
obstacles = [
    {'center': np.array([1.2, 1.0]),  'radius': 0.35},
    {'center': np.array([0.0, 1.4]),  'radius': 0.3},
    {'center': np.array([-0.5, 0.8]), 'radius': 0.3},
]

In [ ]:
def compute_sdf(obstacles, xlim=WS_XLIM, ylim=WS_YLIM, resolution=GRID_RESOLUTION):
    """
    Compute signed distance field on a grid.
    For circular obstacles: d_i(x) = ||x - c_i|| - r_i.
    Composite: d(x) = min_i d_i(x).
    
    Returns: sdf, sdf_grad_x, sdf_grad_y, x_coords, y_coords
    """
    x_coords = np.arange(xlim[0], xlim[1] + resolution, resolution)
    y_coords = np.arange(ylim[0], ylim[1] + resolution, resolution)
    X, Y = np.meshgrid(x_coords, y_coords)
    
    sdf = np.full_like(X, np.inf)
    for obs in obstacles:
        dist = np.sqrt((X - obs['center'][0])**2 + (Y - obs['center'][1])**2) - obs['radius']
        sdf = np.minimum(sdf, dist)
    
    # Gradient via central differences on the composite SDF
    sdf_grad_y, sdf_grad_x = np.gradient(sdf, resolution)
    
    return sdf, sdf_grad_x, sdf_grad_y, x_coords, y_coords

In [ ]:
# Vectorized cost function and its derivative
def obstacle_cost_vectorized(d, epsilon=EPSILON):
    """Piecewise cost: large inside obstacles, decaying in safety margin, zero outside."""
    cost = np.zeros_like(d)
    mask_inside = d < 0
    mask_margin = (d >= 0) & (d <= epsilon)
    cost[mask_inside] = -d[mask_inside] + epsilon / 2.0
    cost[mask_margin] = (d[mask_margin] - epsilon)**2 / (2.0 * epsilon)
    return cost


def obstacle_cost_grad_vectorized(d, epsilon=EPSILON):
    """Derivative of c(d) w.r.t. d."""
    grad = np.zeros_like(d)
    mask_inside = d < 0
    mask_margin = (d >= 0) & (d <= epsilon)
    grad[mask_inside] = -1.0
    grad[mask_margin] = (d[mask_margin] - epsilon) / epsilon
    return grad


# Scalar versions for single-point evaluation
def obstacle_cost_scalar(d, epsilon=EPSILON):
    if d < 0:
        return -d + epsilon / 2.0
    elif d <= epsilon:
        return (d - epsilon)**2 / (2.0 * epsilon)
    return 0.0


def obstacle_cost_grad_scalar(d, epsilon=EPSILON):
    if d < 0:
        return -1.0
    elif d <= epsilon:
        return (d - epsilon) / epsilon
    return 0.0

In [ ]:
# Visualize SDF and cost field
sdf, sdf_gx, sdf_gy, x_coords, y_coords = compute_sdf(obstacles)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# SDF
ax = axes[0]
levels = np.linspace(-1, 2, 30)
cf = ax.contourf(x_coords, y_coords, sdf, levels=levels, cmap='RdYlBu', extend='both')
ax.contour(x_coords, y_coords, sdf, levels=[0], colors='black', linewidths=2)
ax.contour(x_coords, y_coords, sdf, levels=[EPSILON], colors='orange', linewidths=1.5, linestyles='--')
plt.colorbar(cf, ax=ax, shrink=0.8, label='Signed distance $d(x)$')
ax.set_title('Signed Distance Field (SDF)', fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')
ax.set_xlim(-2, 2.5); ax.set_ylim(-1, 2.5)

# Cost field c(d)
ax = axes[1]
cost_field = obstacle_cost_vectorized(sdf)
cf2 = ax.contourf(x_coords, y_coords, cost_field, levels=20, cmap='hot_r')
for obs in obstacles:
    circle = plt.Circle(obs['center'], obs['radius'], fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(circle)
plt.colorbar(cf2, ax=ax, shrink=0.8, label='Cost $c(d(x))$')
ax.set_title(f'Obstacle Cost Field ($\\varepsilon$ = {EPSILON})', fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')
ax.set_xlim(-2, 2.5); ax.set_ylim(-1, 2.5)

plt.tight_layout()
plt.show()

### Obstacle Gradient for a Multi-Link Arm

The obstacle gradient at waypoint $i$ is (simplified, dropping curvature terms):

$$\nabla_{q_i} F_{\text{obs}} \approx \sum_{\text{body points } u} c'(d(x_u)) \cdot \|v_u\| \cdot J_u^T \nabla d(x_u)$$

where:
- $x_u$ = workspace position of body point $u$ (from FK)
- $d(x_u)$ = signed distance at that position
- $c'(d)$ = derivative of the cost function
- $\nabla d(x_u)$ = spatial gradient of the SDF (2D vector)
- $J_u$ = Jacobian at body point $u$ (2 $\times$ 3)
- $\|v_u\| = \|J_u \dot{q}\|$ = velocity magnitude (**arc-length weighting**)
- $\dot{q} \approx (q_{i+1} - q_{i-1}) / 2$ via central differences

In [ ]:
def build_sdf_interpolators(sdf, sdf_grad_x, sdf_grad_y, x_coords, y_coords):
    """Build scipy interpolation functions for SDF and its gradient."""
    sdf_interp = RegularGridInterpolator(
        (y_coords, x_coords), sdf,
        method='linear', bounds_error=False, fill_value=10.0)
    grad_x_interp = RegularGridInterpolator(
        (y_coords, x_coords), sdf_grad_x,
        method='linear', bounds_error=False, fill_value=0.0)
    grad_y_interp = RegularGridInterpolator(
        (y_coords, x_coords), sdf_grad_y,
        method='linear', bounds_error=False, fill_value=0.0)
    return sdf_interp, grad_x_interp, grad_y_interp

In [ ]:
def compute_obstacle_cost_and_gradient(xi, q_start, q_goal,
                                       sdf_interp, grad_x_interp, grad_y_interp,
                                       n_body_points=N_BODY_POINTS, epsilon=EPSILON):
    """
    Compute total obstacle cost and its gradient w.r.t. all waypoints.
    
    Optimized: computes FK once per waypoint, batches all SDF queries per waypoint.
    
    Args:
        xi: (n_waypoints, DOF) trajectory
        sdf_interp, grad_x_interp, grad_y_interp: interpolation functions
    Returns:
        total_cost: scalar
        gradient: (n_waypoints, DOF) array
    """
    n = xi.shape[0]
    gradient = np.zeros_like(xi)
    total_cost = 0.0
    n_bp_total = 3 * n_body_points  # body points per waypoint
    
    traj = full_trajectory(xi, q_start, q_goal)  # (n+2, DOF)
    
    for i in range(n):  # each interior waypoint
        q = traj[i + 1]
        q_vel = (traj[i + 2] - traj[i]) / 2.0
        
        # Compute all body point positions and Jacobians at once (FK computed once)
        bp_positions, bp_jacobians = compute_all_body_data(q, n_body_points)
        
        # Batch SDF queries: all body points for this waypoint in one call
        # RegularGridInterpolator expects (y, x) order
        query_pts = bp_positions[:, ::-1]  # (n_bp_total, 2) in (y, x) order
        d_vals = sdf_interp(query_pts)              # (n_bp_total,)
        grad_dx = grad_x_interp(query_pts)          # (n_bp_total,)
        grad_dy = grad_y_interp(query_pts)          # (n_bp_total,)
        grad_d_all = np.column_stack([grad_dx, grad_dy])  # (n_bp_total, 2)
        
        # Vectorized cost and cost derivative
        c_vals = obstacle_cost_vectorized(d_vals, epsilon)        # (n_bp_total,)
        c_primes = obstacle_cost_grad_vectorized(d_vals, epsilon) # (n_bp_total,)
        
        # Compute velocity magnitudes: v_bp = J @ q_vel for each body point
        # bp_jacobians is (n_bp_total, 2, DOF), q_vel is (DOF,)
        v_bps = bp_jacobians @ q_vel                  # (n_bp_total, 2)
        vel_mags = np.linalg.norm(v_bps, axis=1) + 1e-8  # (n_bp_total,)
        
        # Accumulate cost
        total_cost += np.sum(c_vals * vel_mags) / n_body_points
        
        # Gradient: sum of c'(d) * ||v|| * J^T @ grad_d for each body point
        # J^T @ grad_d: (DOF,) for each body point
        # bp_jacobians is (n_bp_total, 2, DOF), grad_d_all is (n_bp_total, 2)
        # J^T @ grad_d = einsum('b2d,b2->bd', jacobians_transposed, grad_d)
        #              = einsum('b2d,b2->bd', bp_jacobians transposed, grad_d_all)
        # Since J is (2, DOF), J^T @ grad_d = (DOF,), we use:
        Jt_grad_d = np.einsum('bij,bi->bj', bp_jacobians, grad_d_all)  # (n_bp_total, DOF)
        
        # Weight by c' * ||v|| and sum
        weights = (c_primes * vel_mags) / n_body_points  # (n_bp_total,)
        gradient[i] = np.sum(weights[:, np.newaxis] * Jt_grad_d, axis=0)
    
    return total_cost, gradient

In [ ]:
# Build interpolators
sdf_interp, gx_interp, gy_interp = build_sdf_interpolators(
    sdf, sdf_gx, sdf_gy, x_coords, y_coords)

# Verify obstacle gradient with finite differences (on a small subset for speed)
xi_test_obs = init_trajectory(q_start, q_goal, N_WAYPOINTS) + 0.05 * np.random.randn(N_WAYPOINTS, DOF)

_, grad_obs_analytical = compute_obstacle_cost_and_gradient(
    xi_test_obs, q_start, q_goal, sdf_interp, gx_interp, gy_interp)

# Check a few random waypoints
eps_fd = 1e-5
check_indices = np.random.choice(N_WAYPOINTS, size=5, replace=False)
max_rel_err = 0.0
for idx in check_indices:
    for d in range(DOF):
        xi_p = xi_test_obs.copy(); xi_p[idx, d] += eps_fd
        xi_m = xi_test_obs.copy(); xi_m[idx, d] -= eps_fd
        c_p, _ = compute_obstacle_cost_and_gradient(xi_p, q_start, q_goal, sdf_interp, gx_interp, gy_interp)
        c_m, _ = compute_obstacle_cost_and_gradient(xi_m, q_start, q_goal, sdf_interp, gx_interp, gy_interp)
        g_num = (c_p - c_m) / (2 * eps_fd)
        g_ana = grad_obs_analytical[idx, d]
        if abs(g_num) > 1e-8:
            rel_err = abs(g_ana - g_num) / (abs(g_num) + 1e-10)
            max_rel_err = max(max_rel_err, rel_err)

print(f'Obstacle gradient verification (5 random waypoints):')
print(f'  Max relative error: {max_rel_err:.2e}')
print(f'  Approximate match is expected (arc-length term is simplified)')

In [ ]:
# Visualize: arm near obstacles with body point costs
q_demo = np.array([0.8, 0.4, 0.2])  # config near obstacles

fig, ax = plt.subplots(figsize=(8, 8))
# Cost field background
cost_field = obstacle_cost_vectorized(sdf)
ax.contourf(x_coords, y_coords, cost_field, levels=20, cmap='hot_r', alpha=0.4)
for obs in obstacles:
    circle = plt.Circle(obs['center'], obs['radius'], color='red', alpha=0.3)
    ax.add_patch(circle)
    circle_edge = plt.Circle(obs['center'], obs['radius'], fill=False, edgecolor='red', linewidth=2)
    ax.add_patch(circle_edge)

# Draw arm
plot_arm(ax, q_demo, color='steelblue', linewidth=3)

# Body points colored by cost
for link_idx in range(3):
    for bp in range(N_BODY_POINTS):
        u = (bp + 0.5) / N_BODY_POINTS
        x_bp = body_point_position(q_demo, link_idx, u)
        d_val = sdf_interp(np.array([[x_bp[1], x_bp[0]]])).item()
        c_val = obstacle_cost_scalar(d_val)
        color = 'yellow' if c_val > 0.01 else 'lightblue'
        size = 30 + 200 * c_val
        ax.scatter(x_bp[0], x_bp[1], c=[c_val], cmap='hot_r', s=size,
                   vmin=0, vmax=0.5, edgecolors='black', linewidths=0.5, zorder=10)

ax.set_xlim(-1.5, 2.5)
ax.set_ylim(-1, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Body Points Colored by Obstacle Cost')
plt.tight_layout()
plt.show()

---
## 13. Case 2: CHOMP With Obstacles

Now we combine both costs in the full CHOMP algorithm:

```
Initialize: xi = straight_line(q_start, q_goal)
Precompute: K, A, A_inv, b, SDF interpolators
For iteration = 1 to MAX_ITER:
    grad_smooth = A @ xi + b
    cost_obs, grad_obs = obstacle_cost_and_gradient(xi)
    total_grad = grad_obs + lambda * grad_smooth
    xi = xi - (1/eta) * A_inv @ total_grad    # covariant update
```

In [ ]:
def chomp_with_obstacles(xi_init, q_start, q_goal, obstacles,
                         lam=LAMBDA, eta=ETA, max_iter=MAX_ITER,
                         epsilon=EPSILON, n_body_points=N_BODY_POINTS,
                         record_every=5):
    """
    Full CHOMP algorithm with smoothness + obstacle cost.
    
    Returns:
        xi_opt: optimized trajectory
        cost_history: dict with 'total', 'smooth', 'obstacle' lists
        trajectory_history: list of trajectory snapshots
    """
    n = xi_init.shape[0]
    
    # Build smoothness matrices
    K, A, A_inv, e, b_vec = build_smoothness_matrices(n, q_start, q_goal)
    
    # Build SDF and interpolators
    sdf_data, sdf_gx, sdf_gy, xc, yc = compute_sdf(obstacles)
    si, gxi, gyi = build_sdf_interpolators(sdf_data, sdf_gx, sdf_gy, xc, yc)
    
    xi = xi_init.copy()
    cost_history = {'total': [], 'smooth': [], 'obstacle': []}
    trajectory_history = [xi.copy()]
    
    for iteration in range(max_iter):
        # Smoothness
        cost_smooth = compute_smoothness_cost(xi, K, e)
        grad_smooth = compute_smoothness_gradient(xi, A, b_vec)
        
        # Obstacle
        cost_obs, grad_obs = compute_obstacle_cost_and_gradient(
            xi, q_start, q_goal, si, gxi, gyi, n_body_points, epsilon)
        
        # Record costs
        cost_history['total'].append(cost_obs + lam * cost_smooth)
        cost_history['smooth'].append(cost_smooth)
        cost_history['obstacle'].append(cost_obs)
        
        # Total gradient and covariant update
        total_grad = grad_obs + lam * grad_smooth
        xi -= (1.0 / eta) * (A_inv @ total_grad)
        
        if iteration % record_every == 0:
            trajectory_history.append(xi.copy())
    
    trajectory_history.append(xi.copy())
    return xi, cost_history, trajectory_history

In [ ]:
# Run CHOMP with obstacles
xi_init_obs = init_trajectory(q_start, q_goal, N_WAYPOINTS)

print('Running CHOMP with obstacles...')
xi_opt_obs, costs_obs, traj_hist_obs = chomp_with_obstacles(
    xi_init_obs, q_start, q_goal, obstacles,
    lam=LAMBDA, eta=ETA, max_iter=MAX_ITER)
print(f'Done. Final total cost: {costs_obs["total"][-1]:.4f}')
print(f'  Obstacle cost: {costs_obs["obstacle"][-1]:.4f}')
print(f'  Smoothness cost: {costs_obs["smooth"][-1]:.4f}')

In [ ]:
# Cost convergence (3 panels)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = ['Total Cost $U[\\xi]$', 'Obstacle Cost $F_{obs}$', 'Smoothness Cost $F_{smooth}$']
keys = ['total', 'obstacle', 'smooth']
colors_plot = ['navy', 'red', 'steelblue']

for ax, title, key, color in zip(axes, titles, keys, colors_plot):
    ax.plot(costs_obs[key], color=color, linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cost')
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)
    if min(costs_obs[key]) > 0:
        ax.set_yscale('log')

plt.suptitle('CHOMP With Obstacles: Cost Convergence', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Before vs After (static comparison)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

plot_trajectory_workspace(xi_init_obs, q_start, q_goal, ax=axes[0],
                          obstacles=obstacles,
                          title='Initial Trajectory (Straight Line)')
plot_trajectory_workspace(xi_opt_obs, q_start, q_goal, ax=axes[1],
                          obstacles=obstacles,
                          title='Optimized Trajectory (CHOMP)')

plt.tight_layout()
plt.show()

In [ ]:
# Summary figure: cost field + optimized trajectory
fig, ax = plt.subplots(figsize=(10, 10))

# Cost field background
cost_field = obstacle_cost_vectorized(sdf)
ax.contourf(x_coords, y_coords, cost_field, levels=20, cmap='hot_r', alpha=0.3)

# Obstacles
for obs in obstacles:
    circle = plt.Circle(obs['center'], obs['radius'], color='red', alpha=0.4)
    ax.add_patch(circle)
    circle_edge = plt.Circle(obs['center'], obs['radius'],
                            fill=False, edgecolor='darkred', linewidth=2)
    ax.add_patch(circle_edge)

# Draw optimized arm at several waypoints
traj_full = full_trajectory(xi_opt_obs, q_start, q_goal)
n_show = 10
show_idx = np.linspace(0, len(traj_full) - 1, n_show, dtype=int)
for idx in show_idx:
    alpha = 0.15 + 0.7 * (idx / (len(traj_full) - 1))
    plot_arm(ax, traj_full[idx], color='steelblue', alpha=alpha, linewidth=2, markersize=4)

# Start and goal
plot_arm(ax, q_start, color='green', linewidth=3.5, markersize=9, label='Start')
plot_arm(ax, q_goal, color='red', linewidth=3.5, markersize=9, label='Goal')

# End-effector path
ee_path = np.array([forward_kinematics(q)[3] for q in traj_full])
ax.plot(ee_path[:, 0], ee_path[:, 1], '-', color='navy', linewidth=2.5, alpha=0.9, label='EE path')

ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.0, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=12)
ax.set_title('CHOMP Result: Optimized Trajectory Avoiding Obstacles', fontsize=14)
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

In [ ]:
# Animation: trajectory evolution
def animate_chomp(traj_hist, q_start, q_goal, obstacles, interval=300):
    """Animate CHOMP trajectory evolution."""
    fig, ax = plt.subplots(figsize=(9, 9))
    
    def draw_frame(frame_idx):
        ax.clear()
        
        # Cost field
        cost_field_local = obstacle_cost_vectorized(sdf)
        ax.contourf(x_coords, y_coords, cost_field_local, levels=20, cmap='hot_r', alpha=0.25)
        
        # Obstacles
        for obs in obstacles:
            circle = plt.Circle(obs['center'], obs['radius'], color='red', alpha=0.4)
            ax.add_patch(circle)
            ce = plt.Circle(obs['center'], obs['radius'], fill=False, edgecolor='darkred', linewidth=2)
            ax.add_patch(ce)
        
        xi_frame = traj_hist[frame_idx]
        traj_f = full_trajectory(xi_frame, q_start, q_goal)
        
        # Arms at waypoints
        n_arms = 8
        idx_show = np.linspace(0, len(traj_f) - 1, n_arms, dtype=int)
        for idx in idx_show:
            alpha = 0.15 + 0.65 * (idx / (len(traj_f) - 1))
            plot_arm(ax, traj_f[idx], color='steelblue', alpha=alpha, linewidth=2, markersize=4)
        
        plot_arm(ax, q_start, color='green', linewidth=3, markersize=8)
        plot_arm(ax, q_goal, color='red', linewidth=3, markersize=8)
        
        # EE path
        ee = np.array([forward_kinematics(q)[3] for q in traj_f])
        ax.plot(ee[:, 0], ee[:, 1], '-', color='navy', linewidth=2, alpha=0.8)
        
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.0, 2.5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        iteration = frame_idx * 5  # since we record every 5 iterations
        ax.set_title(f'CHOMP Evolution — Iteration {iteration}', fontsize=13)
    
    anim = animation.FuncAnimation(fig, draw_frame, frames=len(traj_hist), interval=interval)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_chomp(traj_hist_obs, q_start, q_goal, obstacles)

---
## 14. Analysis

Let's study how CHOMP's behavior changes with different parameter settings.

In [ ]:
# Lambda sensitivity: smoothness weight
lambdas = [0.5, 5.0, 10.0, 50.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(5 * len(lambdas), 6))

for i, lam_val in enumerate(lambdas):
    xi_opt_lam, _, _ = chomp_with_obstacles(
        xi_init_obs, q_start, q_goal, obstacles, lam=lam_val, max_iter=300)
    plot_trajectory_workspace(xi_opt_lam, q_start, q_goal, ax=axes[i],
                              obstacles=obstacles,
                              title=f'$\\lambda$ = {lam_val}')

plt.suptitle('Effect of Smoothness Weight $\\lambda$', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('Low lambda → aggressive obstacle avoidance (potentially jerky)')
print('High lambda → very smooth but may clip obstacles')

In [ ]:
# Step size sensitivity
etas = [20, 50, 100, 500]
fig, ax = plt.subplots(figsize=(10, 6))

for eta_val in etas:
    _, costs_eta, _ = chomp_with_obstacles(
        xi_init_obs, q_start, q_goal, obstacles, eta=eta_val, max_iter=200)
    ax.plot(costs_eta['total'], linewidth=2, label=f'$\\eta$ = {eta_val}')

ax.set_xlabel('Iteration')
ax.set_ylabel('Total Cost')
ax.set_title('Convergence for Different Step Sizes $\\eta$', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()
print('Small eta → larger steps → faster convergence but risk of oscillation')
print('Large eta → smaller steps → stable but slow convergence')

In [ ]:
# Local minima demonstration: different initializations → different solutions
xi_init_1 = init_trajectory(q_start, q_goal, N_WAYPOINTS).copy()

# Second initialization: add a bias to bend the trajectory differently
xi_init_2 = init_trajectory(q_start, q_goal, N_WAYPOINTS).copy()
t_param = np.linspace(0, 1, N_WAYPOINTS)
xi_init_2[:, 0] += 0.8 * np.sin(np.pi * t_param)   # bias joint 1 upward
xi_init_2[:, 1] -= 0.5 * np.sin(np.pi * t_param)   # bias joint 2

xi_opt_1, costs_1, _ = chomp_with_obstacles(xi_init_1, q_start, q_goal, obstacles, max_iter=300)
xi_opt_2, costs_2, _ = chomp_with_obstacles(xi_init_2, q_start, q_goal, obstacles, max_iter=300)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
plot_trajectory_workspace(xi_opt_1, q_start, q_goal, ax=axes[0],
                          obstacles=obstacles,
                          title=f'Initialization 1 (cost={costs_1["total"][-1]:.2f})')
plot_trajectory_workspace(xi_opt_2, q_start, q_goal, ax=axes[1],
                          obstacles=obstacles,
                          title=f'Initialization 2 (cost={costs_2["total"][-1]:.2f})')

plt.suptitle('Local Minima: Different Initializations → Different Solutions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('CHOMP is a local optimizer — the initial trajectory matters!')
print('Hamiltonian Monte Carlo (HMC) can help escape local minima by adding random momentum.')

---
## 15. Limitations & Extensions

### Limitations
- **Local minima:** CHOMP uses gradient descent, so it converges to the nearest local minimum. The solution depends on initialization.
- **Hamiltonian Monte Carlo (HMC):** The original CHOMP paper proposes HMC to escape local minima. It adds random momentum $\gamma$ and simulates Hamiltonian dynamics: $H(\xi, \gamma) = U(\xi) + K(\gamma)$, allowing the trajectory to "roll" out of shallow minima.
- **Computational cost:** The body-point loop is $O(n \cdot B \cdot d)$ per iteration, which becomes expensive for high-DOF robots.
- **Joint limits:** Not handled here — would require projecting waypoints back into feasible joint ranges.

### Related Algorithms
| Algorithm | Key Difference |
|-----------|---------------|
| **STOMP** | Stochastic — uses random noise instead of gradients. Doesn't need gradient computation. |
| **TrajOpt** | Sequential convex optimization — uses convex approximations with trust regions. More robust to local minima. |
| **GPMP2** | Gaussian Process-based — models trajectories as GP priors. Elegant probabilistic formulation. |

### Key Takeaways
1. CHOMP elegantly combines physics-inspired smoothness with obstacle avoidance
2. The **covariant gradient** ($A^{-1}$ smoothing) is the key insight — it prevents kinks
3. **Arc-length parametrization** ensures the obstacle cost depends on path geometry, not timing
4. The algorithm is simple to implement but powerful in practice